# Benchmark DEFINITIVO: HALO-S v2.0 vs Transformer (70M params)

## Comparacion a Escala Real en WikiText-103

Este es el benchmark **definitivo v2.0** que compara HALO-S contra un Transformer
denso a escala significativa (~70M parametros).

**Optimizaciones v2.0:**
- SwiGLU FFN (activacion superior a GELU)
- Gradient Checkpointing (reduce memoria de activaciones ~50%)
- Mixed Precision (FP16 forward/backward)
- Gradient Accumulation (batch efectivo = 32)
- GQA 4:1 ratio (eficiencia de KV cache)

**Entorno:** Google Kaggle con 2x T4 GPUs (30GB VRAM total)

**Dataset:** WikiText-103-raw-v1 COMPLETO (~500M caracteres)

**Arquitectura (~70M params):**
- hidden_size=768, num_layers=12, num_heads=12, head_dim=64
- seq_len=1024 (umbral SDPA)
- 2 epocas de entrenamiento

**Metricas comparadas:**
- Tiempo de entrenamiento
- Perdida de validacion / Perplejidad
- Velocidad de generacion (200 tokens)
- Latencia de forward pass (256, 512, 1024)
- Uso de memoria GPU
- Calidad de generacion (3 prompts)

## 1. Instalacion de Dependencias

In [ ]:
!pip install pyhalos==2.0.0 datasets tqdm --quiet

## 2. Verificacion de GPU

In [ ]:
import torch
import sys

print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

total_memory = 0
if torch.cuda.is_available():
    print(f'Numero de GPUs: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1e9
        total_memory += mem_gb
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
        print(f'    Memoria: {mem_gb:.1f} GB')
    print(f'\n  Memoria GPU TOTAL: {total_memory:.1f} GB')
else:
    print('ADVERTENCIA: No se detecto GPU. Este notebook requiere CUDA.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsando dispositivo: {device}')

## 3. Importaciones

In [ ]:
import time
import math
import numpy as np
from tqdm import tqdm
from datasets import load_dataset

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.amp  # Mixed precision

# Importaciones de HALO-S v2.0
from halo import HaloConfig, HaloSModel, CharacterTokenizer, set_seed
from halo.models.baseline_model import BaselineModel
from halo.utils.metrics import count_parameters

# Semilla para reproducibilidad
set_seed(42)
print('Todas las importaciones exitosas (v2.0)')

## 4. Descarga y Procesamiento del Dataset

Usamos **WikiText-103** (wikitext-103-raw-v1), un dataset mucho mas grande que WikiText-2.
Contiene ~500M caracteres del split de entrenamiento COMPLETO.
Tokenizamos a nivel de caracter con `CharacterTokenizer` (vocab_size=256).

In [ ]:
# Descargar WikiText-103 (dataset completo)
dataset = load_dataset('wikitext', 'wikitext-103-raw-v1')
print(f'Train: {len(dataset["train"])} ejemplos')
print(f'Validation: {len(dataset["validation"])} ejemplos')
print(f'\nEjemplo de texto:')
print(dataset['train'][100]['text'][:200])

### 4.1 Tokenizacion y Preparacion de Chunks (seq_len=1024)

In [ ]:
# Configuracion v2.0 para 70M params
MAX_SEQ_LEN = 1024
BATCH_SIZE = 2         # Split a 1 por GPU via DataParallel
GRAD_ACCUM = 8         # Effective batch = 2 * 8 * 2 GPUs = 32
EPOCHS = 2
LR = 3e-4

# Tokenizador a nivel de caracter
tokenizer = CharacterTokenizer()
print(f'Vocab size: {tokenizer.vocab_size}')

def tokenize_and_chunk(split_data, max_len):
    """Tokeniza el texto, concatena y divide en chunks de tamano fijo."""
    # Filtrar lineas vacias y concatenar todo el texto
    all_text = '\n'.join([item['text'] for item in split_data if item['text'].strip()])
    print(f'  Texto total: {len(all_text):,} caracteres')
    
    # Tokenizar
    all_tokens = tokenizer.encode(all_text)
    print(f'  Tokens totales: {len(all_tokens):,}')
    
    # Dividir en chunks de max_len + 1 (para x, y shift)
    chunks = []
    for i in range(0, len(all_tokens) - max_len, max_len):
        chunk = all_tokens[i:i + max_len + 1]
        if len(chunk) == max_len + 1:
            chunks.append(chunk)
    
    print(f'  Chunks generados: {len(chunks):,}')
    return chunks

print('Procesando train (WikiText-103 COMPLETO)...')
train_chunks = tokenize_and_chunk(dataset['train'], MAX_SEQ_LEN)
print('\nProcesando validation...')
val_chunks = tokenize_and_chunk(dataset['validation'], MAX_SEQ_LEN)

### 4.2 Dataset de PyTorch

In [ ]:
class WikiTextCharDataset(Dataset):
    """Dataset que retorna pares (x, y) con shift de 1 posicion."""
    
    def __init__(self, chunks):
        self.chunks = chunks
    
    def __len__(self):
        return len(self.chunks)
    
    def __getitem__(self, idx):
        chunk = torch.tensor(self.chunks[idx], dtype=torch.long)
        x = chunk[:-1]  # input: tokens 0..N-1
        y = chunk[1:]   # target: tokens 1..N
        return x, y

train_dataset = WikiTextCharDataset(train_chunks)
val_dataset = WikiTextCharDataset(val_chunks)

print(f'Train dataset: {len(train_dataset):,} muestras')
print(f'Val dataset: {len(val_dataset):,} muestras')
print(f'\nEstimacion de steps por epoca: {len(train_dataset) // BATCH_SIZE:,}')
print(f'Optimizer steps por epoca: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM):,}')

# Verificar forma
x, y = train_dataset[0]
print(f'\nForma de x: {x.shape}  (seq_len={MAX_SEQ_LEN})')
print(f'Forma de y: {y.shape}')

## 5. Creacion de Modelos (~70M parametros)

### Arquitectura:
- **HALO-S v2.0**: SwiGLU FFN + GQA 4:1 + Gradient Checkpointing + Atencion Dispersa
- **Transformer Denso**: MHA completo + GELU FFN estandar + Atencion O(N^2)

**Nota:** HALO-S con SwiGLU tendra mas parametros que el baseline (3 matrices vs 2 en FFN).
Esto es por diseno — la comparacion clave es VELOCIDAD y MEMORIA, no conteo de params.

In [ ]:
# ============================================
# Modelo A: HALO-S v2.0 (Atencion Dispersa + SwiGLU)
# ============================================
halo_config = HaloConfig(
    vocab_size=256,
    hidden_size=768,
    num_layers=12,
    num_heads=12,
    num_kv_heads=3,       # GQA 4:1 ratio
    num_globals=2,
    local_window=64,
    dilated_offsets=[1, 2, 4, 8, 16, 32, 64, 128],
    num_random=2,
    dropout=0.1,
    max_seq_len=1024,
    use_swiglu=True,      # v2.0 SwiGLU
)
halo_model = HaloSModel(halo_config)
halo_model.enable_gradient_checkpointing()  # v2.0 memory optimization

# ============================================
# Modelo B: Transformer Denso (Baseline)
# ============================================
baseline_config = HaloConfig(
    vocab_size=256,
    hidden_size=768,
    num_layers=12,
    num_heads=12,
    num_kv_heads=12,      # MHA (no GQA)
    num_globals=2,
    local_window=64,
    dilated_offsets=[1, 2, 4, 8, 16, 32, 64, 128],
    num_random=2,
    dropout=0.1,
    max_seq_len=1024,
    use_swiglu=False,     # Standard GELU (BaselineModel usa su propio FeedForward)
)
baseline_model = BaselineModel(baseline_config)

# Comparar parametros
halo_params = count_parameters(halo_model)
baseline_params = count_parameters(baseline_model)

print('=' * 60)
print('COMPARACION DE MODELOS (70M scale)')
print('=' * 60)
print(f'HALO-S v2.0  | Parametros: {halo_params:>12,}')
print(f'Transformer  | Parametros: {baseline_params:>12,}')
print(f'Diferencia   | HALO-S tiene {halo_params - baseline_params:,} params extra (SwiGLU)')
print('=' * 60)
print(f'\nHALO-S v2.0 features:')
print(f'  - SwiGLU FFN: SI')
print(f'  - GQA ratio: 4:1 (12 Q heads, 3 KV heads)')
print(f'  - Gradient Checkpointing: ACTIVO')
print(f'  - Atencion: Dispersa O(N*K)')
print(f'\nTransformer Baseline:')
print(f'  - FFN: GELU estandar (2 matrices)')
print(f'  - Atencion: MHA completo (12 KV heads)')
print(f'  - Atencion: Densa O(N^2)')

## 6. Entrenamiento v2.0: DataParallel + Mixed Precision + Gradient Accumulation

**Estrategia de memoria para 2x T4 (15GB cada una):**
- batch_size=2 (split 1 por GPU via DataParallel)
- gradient_accumulation=8 (batch efectivo = 2 x 8 x 2 GPUs = 32)
- Mixed Precision FP16 (reduce memoria ~50%)
- Gradient Checkpointing en HALO-S (ventaja v2.0)

**Hiperparametros LLM estandar:**
- AdamW: lr=3e-4, weight_decay=0.1, betas=(0.9, 0.95)
- Gradient clipping: 1.0

### 6.0 Funcion de Entrenamiento v2.0

In [ ]:
def train_model_v2(model, train_dataset, val_dataset, epochs, batch_size, lr, model_name, grad_accum=8):
    """Entrenamiento v2.0 con DataParallel + Mixed Precision + Gradient Accumulation."""
    if torch.cuda.device_count() > 1:
        print(f'  Usando {torch.cuda.device_count()} GPUs con DataParallel')
        model_dp = nn.DataParallel(model)
    else:
        model_dp = model
    
    model_dp = model_dp.cuda()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1, betas=(0.9, 0.95))
    
    # Mixed precision scaler
    scaler = torch.amp.GradScaler('cuda')
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size,
        num_workers=2, pin_memory=True
    )
    
    effective_batch = batch_size * grad_accum * torch.cuda.device_count()
    print(f'  Batch size efectivo: {effective_batch}')
    print(f'  Steps por epoca: {len(train_loader):,}')
    print(f'  Optimizer steps por epoca: {len(train_loader) // grad_accum:,}')
    
    history = []
    
    for epoch in range(epochs):
        model_dp.train()
        total_loss = 0
        num_batches = 0
        optimizer.zero_grad()
        
        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f'{model_name} Epoch {epoch+1}/{epochs}')
        
        for step, (x, y) in pbar:
            x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
            
            # Mixed precision forward
            with torch.amp.autocast('cuda'):
                logits, loss = model_dp(x, targets=y)
                if loss.dim() > 0:
                    loss = loss.mean()
                loss = loss / grad_accum
            
            # Scaled backward
            scaler.scale(loss).backward()
            
            total_loss += loss.item() * grad_accum
            num_batches += 1
            
            # Optimizer step every grad_accum steps
            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            if num_batches % 100 == 0:
                pbar.set_postfix({'loss': f'{total_loss/num_batches:.4f}'})
        
        # Flush remaining gradients
        if (step + 1) % grad_accum != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        avg_train_loss = total_loss / num_batches
        
        # Validation (also in mixed precision for speed)
        model_dp.eval()
        val_loss_total = 0
        val_batches = 0
        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                for x, y in tqdm(val_loader, desc=f'{model_name} Val'):
                    x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
                    _, loss = model_dp(x, targets=y)
                    if loss.dim() > 0:
                        loss = loss.mean()
                    val_loss_total += loss.item()
                    val_batches += 1
        
        avg_val_loss = val_loss_total / val_batches
        ppl = math.exp(avg_val_loss)
        print(f'  Epoch {epoch+1} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | PPL: {ppl:.2f}')
        history.append({'epoch': epoch+1, 'train_loss': avg_train_loss, 'val_loss': avg_val_loss})
    
    return history

### 6.1 Entrenamiento de HALO-S v2.0 (2 epocas)

In [ ]:
# HALO-S v2.0 Training
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

print('Entrenando HALO-S v2.0 (SwiGLU + GradCheckpoint + MixedPrecision)...')
print('=' * 60)

halo_start_time = time.time()
halo_history = train_model_v2(
    halo_model, train_dataset, val_dataset,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    model_name='HALO-S v2.0', grad_accum=GRAD_ACCUM
)
torch.cuda.synchronize()
halo_total_time = time.time() - halo_start_time
halo_peak_memory = torch.cuda.max_memory_allocated() / (1024**3)

print(f'\nHALO-S v2.0 entrenado en {halo_total_time:.1f}s ({halo_total_time/60:.1f} min)')
print(f'Pico de memoria GPU: {halo_peak_memory:.2f} GB')

### 6.2 Entrenamiento del Transformer Denso (2 epocas)

In [ ]:
# Baseline Transformer Training
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

print('Entrenando Transformer Denso (MHA + GELU + O(N^2))...')
print('=' * 60)

baseline_start_time = time.time()
baseline_history = train_model_v2(
    baseline_model, train_dataset, val_dataset,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    model_name='Transformer', grad_accum=GRAD_ACCUM
)
torch.cuda.synchronize()
baseline_total_time = time.time() - baseline_start_time
baseline_peak_memory = torch.cuda.max_memory_allocated() / (1024**3)

print(f'\nTransformer Denso entrenado en {baseline_total_time:.1f}s ({baseline_total_time/60:.1f} min)')
print(f'Pico de memoria GPU: {baseline_peak_memory:.2f} GB')

### 6.3 Preparar modelos para inferencia (single GPU)

In [ ]:
# Despues de DataParallel, mover modelos a single GPU para benchmarks de inferencia
halo_model = halo_model.cuda()
baseline_model = baseline_model.cuda()
torch.cuda.empty_cache()
print('Modelos movidos a single GPU para benchmarks de inferencia')

## 7. Guardar Modelos Entrenados

In [ ]:
# Guardar HALO-S v2.0
halo_model.save('halo_s_v2_70m_wikitext103.pt')
print('HALO-S v2.0 guardado: halo_s_v2_70m_wikitext103.pt')

# Guardar Transformer Denso
torch.save(baseline_model.state_dict(), 'transformer_70m_wikitext103.pt')
print('Transformer guardado: transformer_70m_wikitext103.pt')

## 8. Comparacion de Resultados de Entrenamiento

### 8.1 Tiempo de Entrenamiento

In [ ]:
print('=' * 70)
print(f'{"METRICA":<30} {"HALO-S v2.0":>15} {"Transformer":>15} {"Ganancia":>10}')
print('=' * 70)

# Tiempo total
speedup = baseline_total_time / halo_total_time if halo_total_time > 0 else 0
print(f'{"Tiempo total (s)":<30} {halo_total_time:>15.1f} {baseline_total_time:>15.1f} {speedup:>9.2f}x')

# Tiempo por epoca
halo_per_epoch = halo_total_time / EPOCHS
baseline_per_epoch = baseline_total_time / EPOCHS
print(f'{"Tiempo por epoca (s)":<30} {halo_per_epoch:>15.1f} {baseline_per_epoch:>15.1f} {speedup:>9.2f}x')

# Tiempo por epoca en minutos
print(f'{"Tiempo por epoca (min)":<30} {halo_per_epoch/60:>15.1f} {baseline_per_epoch/60:>15.1f} {speedup:>9.2f}x')

# Memoria
mem_ratio = baseline_peak_memory / halo_peak_memory if halo_peak_memory > 0 else 0
print(f'{"Pico memoria GPU (GB)":<30} {halo_peak_memory:>15.2f} {baseline_peak_memory:>15.2f} {mem_ratio:>9.2f}x')

print('=' * 70)

### 8.2 Perdida y Perplejidad

In [ ]:
# Extraer metricas finales
halo_final_train_loss = halo_history[-1]['train_loss']
halo_final_val_loss = halo_history[-1]['val_loss']
baseline_final_train_loss = baseline_history[-1]['train_loss']
baseline_final_val_loss = baseline_history[-1]['val_loss']

halo_perplexity = math.exp(halo_final_val_loss)
baseline_perplexity = math.exp(baseline_final_val_loss)

print('=' * 70)
print(f'{"METRICA":<30} {"HALO-S v2.0":>15} {"Transformer":>15}')
print('=' * 70)
print(f'{"Train Loss (final)":<30} {halo_final_train_loss:>15.4f} {baseline_final_train_loss:>15.4f}')
print(f'{"Val Loss (final)":<30} {halo_final_val_loss:>15.4f} {baseline_final_val_loss:>15.4f}')
print(f'{"Val Perplexity":<30} {halo_perplexity:>15.2f} {baseline_perplexity:>15.2f}')
print('=' * 70)

# Tabla de loss por epoca
print(f'\n{"Epoca":<8} {"HALO Train":<12} {"HALO Val":<12} {"Trans Train":<12} {"Trans Val":<12}')
print('-' * 56)
for h, b in zip(halo_history, baseline_history):
    print(f'{h["epoch"]:<8} {h["train_loss"]:<12.4f} {h["val_loss"]:<12.4f} {b["train_loss"]:<12.4f} {b["val_loss"]:<12.4f}')

## 9. Velocidad de Generacion (200 tokens)

Generamos 200 tokens desde el mismo prompt y medimos el tiempo de cada modelo.

In [ ]:
# Prompt de prueba
prompt_text = 'The history of '
prompt_tokens = tokenizer.encode(prompt_text)
NUM_GENERATE = 200

print(f'Prompt: "{prompt_text}"')
print(f'Tokens del prompt: {len(prompt_tokens)}')
print(f'Tokens a generar: {NUM_GENERATE}')
print('\n' + '=' * 60)

# --- HALO-S v2.0 Generacion ---
halo_model.eval()
halo_input = torch.tensor([prompt_tokens], dtype=torch.long).cuda()

# Warmup
with torch.no_grad():
    _ = halo_model(halo_input)
torch.cuda.synchronize()

torch.cuda.synchronize()
gen_start = time.time()
with torch.no_grad():
    halo_output = halo_model.generate(
        halo_input, max_new_tokens=NUM_GENERATE, temperature=0.8, top_k=50
    )
torch.cuda.synchronize()
halo_gen_time = time.time() - gen_start

halo_gen_text = tokenizer.decode(halo_output[0].tolist())
halo_tokens_per_sec = NUM_GENERATE / halo_gen_time

print(f'\n[HALO-S v2.0] Generacion:')
print(f'  Tiempo: {halo_gen_time:.3f}s')
print(f'  Velocidad: {halo_tokens_per_sec:.1f} tokens/s')
print(f'  Texto generado:')
print(f'  "{halo_gen_text[:300]}"')

# --- Transformer Generacion ---
from halo.generation.samplers import generate as gen_fn

baseline_model.eval()
baseline_input = torch.tensor([prompt_tokens], dtype=torch.long).cuda()

# Warmup
with torch.no_grad():
    _ = baseline_model(baseline_input)
torch.cuda.synchronize()

torch.cuda.synchronize()
gen_start = time.time()
with torch.no_grad():
    baseline_output = gen_fn(
        baseline_model, baseline_input, max_new_tokens=NUM_GENERATE,
        temperature=0.8, top_k=50
    )
torch.cuda.synchronize()
baseline_gen_time = time.time() - gen_start

baseline_gen_text = tokenizer.decode(baseline_output[0].tolist())
baseline_tokens_per_sec = NUM_GENERATE / baseline_gen_time

print(f'\n[Transformer] Generacion:')
print(f'  Tiempo: {baseline_gen_time:.3f}s')
print(f'  Velocidad: {baseline_tokens_per_sec:.1f} tokens/s')
print(f'  Texto generado:')
print(f'  "{baseline_gen_text[:300]}"')

gen_speedup = halo_tokens_per_sec / baseline_tokens_per_sec if baseline_tokens_per_sec > 0 else 0
print(f'\nSpeedup de generacion HALO-S v2.0: {gen_speedup:.2f}x')

## 10. Latencia de Forward Pass

Medimos la latencia del forward pass a longitudes de secuencia: 256, 512, 1024.

In [ ]:
seq_lengths = [256, 512, 1024]
num_warmup = 5
num_trials = 20

halo_latencies = {}
baseline_latencies = {}

halo_model.eval()
baseline_model.eval()

for seq_len in seq_lengths:
    # Input aleatorio
    test_input = torch.randint(0, 256, (1, seq_len), device='cuda')
    
    # --- HALO-S v2.0 ---
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = halo_model(test_input)
    torch.cuda.synchronize()
    
    times = []
    for _ in range(num_trials):
        torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            _ = halo_model(test_input)
        torch.cuda.synchronize()
        times.append(time.time() - start)
    halo_latencies[seq_len] = np.mean(times) * 1000  # ms
    
    # --- Transformer ---
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = baseline_model(test_input)
    torch.cuda.synchronize()
    
    times = []
    for _ in range(num_trials):
        torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            _ = baseline_model(test_input)
        torch.cuda.synchronize()
        times.append(time.time() - start)
    baseline_latencies[seq_len] = np.mean(times) * 1000  # ms

print('=' * 70)
print(f'{"Seq Length":<12} {"HALO-S v2.0 (ms)":>18} {"Transformer (ms)":>18} {"Speedup":>10}')
print('=' * 70)
for sl in seq_lengths:
    sp = baseline_latencies[sl] / halo_latencies[sl] if halo_latencies[sl] > 0 else 0
    print(f'{sl:<12} {halo_latencies[sl]:>18.2f} {baseline_latencies[sl]:>18.2f} {sp:>9.2f}x')
print('=' * 70)

## 11. Calidad de Generacion

Comparamos las muestras generadas por ambos modelos con 3 prompts diferentes.

In [ ]:
prompts = [
    'The United States ',
    'In the year 2024, ',
    'Machine learning is ',
]

print('=' * 70)
print('COMPARACION DE CALIDAD DE GENERACION (70M params)')
print('=' * 70)

for prompt in prompts:
    prompt_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long).cuda()
    
    with torch.no_grad():
        halo_out = halo_model.generate(
            prompt_ids, max_new_tokens=150, temperature=0.7, top_k=40
        )
        base_out = gen_fn(
            baseline_model, prompt_ids, max_new_tokens=150, temperature=0.7, top_k=40
        )
    
    halo_text = tokenizer.decode(halo_out[0].tolist())
    base_text = tokenizer.decode(base_out[0].tolist())
    
    print(f'\nPrompt: "{prompt}"')
    print(f'\n  [HALO-S v2.0]:')
    print(f'  {halo_text[:200]}')
    print(f'\n  [Transformer]:')
    print(f'  {base_text[:200]}')
    print('-' * 70)

## 12. Analisis Detallado de Memoria

Medimos el uso de memoria GPU durante forward pass con batch sizes 1 y 2.

In [ ]:
batch_sizes_test = [1, 2]

print('=' * 70)
print(f'{"Batch Size":<12} {"HALO-S v2.0 (MB)":>18} {"Transformer (MB)":>18} {"Ahorro":>10}')
print('=' * 70)

for bs in batch_sizes_test:
    test_input = torch.randint(0, 256, (bs, MAX_SEQ_LEN), device='cuda')
    
    # HALO-S v2.0
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    with torch.no_grad():
        _ = halo_model(test_input)
    torch.cuda.synchronize()
    halo_mem = torch.cuda.max_memory_allocated() / (1024**2)  # MB
    
    # Transformer
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    with torch.no_grad():
        _ = baseline_model(test_input)
    torch.cuda.synchronize()
    base_mem = torch.cuda.max_memory_allocated() / (1024**2)  # MB
    
    ahorro = (1 - halo_mem / base_mem) * 100 if base_mem > 0 else 0
    print(f'{bs:<12} {halo_mem:>18.1f} {base_mem:>18.1f} {ahorro:>8.1f}%')

print('=' * 70)

## 13. Resumen Final

Tabla consolidada con todos los resultados del benchmark definitivo v2.0.

In [ ]:
print()
print('=' * 75)
print(' RESUMEN BENCHMARK DEFINITIVO: HALO-S v2.0 vs TRANSFORMER'.center(75))
print(' WikiText-103 | 70M params | 2 Epochs | 2x T4 GPUs'.center(75))
print('=' * 75)
print(f'{"Metrica":<35}{"HALO-S v2.0":>18}{"Transformer":>18}')
print('-' * 71)
print(f'{"Parametros":<35}{halo_params:>18,}{baseline_params:>18,}')
print(f'{"Tiempo total (min)":<35}{halo_total_time/60:>18.1f}{baseline_total_time/60:>18.1f}')
print(f'{"Tiempo por epoca (min)":<35}{halo_per_epoch/60:>18.1f}{baseline_per_epoch/60:>18.1f}')
print(f'{"Train Loss (final)":<35}{halo_final_train_loss:>18.4f}{baseline_final_train_loss:>18.4f}')
print(f'{"Val Loss (final)":<35}{halo_final_val_loss:>18.4f}{baseline_final_val_loss:>18.4f}')
print(f'{"Val Perplexity":<35}{halo_perplexity:>18.2f}{baseline_perplexity:>18.2f}')
print(f'{"Pico Memoria GPU (GB)":<35}{halo_peak_memory:>18.2f}{baseline_peak_memory:>18.2f}')
print(f'{"Generacion (tokens/s)":<35}{halo_tokens_per_sec:>18.1f}{baseline_tokens_per_sec:>18.1f}')
print(f'{"Latencia @1024 (ms)":<35}{halo_latencies[1024]:>18.2f}{baseline_latencies[1024]:>18.2f}')
print('-' * 71)
print(f'{"Speedup Entrenamiento":<35}{speedup:>18.2f}x')
print(f'{"Speedup Generacion":<35}{gen_speedup:>18.2f}x')
lat_speedup_1024 = baseline_latencies[1024] / halo_latencies[1024] if halo_latencies[1024] > 0 else 0
print(f'{"Speedup Latencia @1024":<35}{lat_speedup_1024:>18.2f}x')
print('=' * 75)

print('\n' + '=' * 75)
print(' INTERPRETACION DE RESULTADOS'.center(75))
print('=' * 75)

# Perplejidad
if halo_perplexity <= baseline_perplexity * 1.1:
    print('  [OK] HALO-S v2.0 alcanza perplejidad comparable al transformer denso')
else:
    ppl_diff = ((halo_perplexity / baseline_perplexity) - 1) * 100
    print(f'  [!] Transformer tiene {ppl_diff:.1f}% mejor PPL (esperado: atencion completa vs dispersa)')

# Velocidad
if speedup > 1.0:
    print(f'  [OK] HALO-S v2.0 es {speedup:.2f}x mas rapido en entrenamiento')
else:
    print(f'  [!] Transformer ligeramente mas rapido en entrenamiento')

# Memoria
if halo_peak_memory < baseline_peak_memory:
    ahorro_mem = (1 - halo_peak_memory / baseline_peak_memory) * 100
    print(f'  [OK] HALO-S v2.0 usa {ahorro_mem:.1f}% menos memoria GPU (grad checkpoint + sparse)')
else:
    print(f'  [!] Uso de memoria similar entre ambos modelos')

# Generacion
if gen_speedup > 1.0:
    print(f'  [OK] HALO-S v2.0 genera texto {gen_speedup:.2f}x mas rapido')
else:
    print(f'  [!] Velocidad de generacion similar')

# Latencia
if lat_speedup_1024 > 1.0:
    print(f'  [OK] Forward pass {lat_speedup_1024:.2f}x mas rapido a seq_len=1024')
else:
    print(f'  [!] Latencia similar a seq_len=1024')

print('\n' + '=' * 75)
print(' VENTAJAS v2.0 vs v1.0'.center(75))
print('=' * 75)
print('  - SwiGLU FFN: mejor convergencia y representacion')
print('  - Gradient Checkpointing: permite 70M params en T4 (15GB)')
print('  - Mixed Precision: ~2x throughput, ~50% menos memoria')
print('  - GQA 4:1: reduce KV cache sin perdida de calidad')
print('  - WikiText-103: 50x mas datos que WikiText-2')
print('  - seq_len=1024: beneficios reales de atencion dispersa O(N*K)')
print('=' * 75)